<a href="https://colab.research.google.com/github/azhgh22/SentimentAnalysis/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
user_name = userdata.get('GITHUB_USERNAME')
mail = userdata.get('GITHUB_MAIL')

!git config --global user.name "{user_name}"
!git config --global user.email "{mail}"
!git clone https://{token}@github.com/azhgh22/SentimentAnalysis.git
! pip install kaggle
! mkdir ~/.kaggle
! cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json
! kaggle competitions download -c sentiment-analysis-on-movie-reviews
! unzip /content/sentiment-analysis-on-movie-reviews.zip -d /content
! unzip /content/train.tsv.zip -d /content
! unzip /content/test.tsv.zip -d /content
! rm -rf /content/sentiment-analysis-on-movie-reviews.zip
! rm -rf /content/train.tsv.zip
! rm -rf /content/test.tsv.zip

# **Imports**

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras.preprocessing.sequence import pad_sequences
from torch.nn.utils.rnn import pack_padded_sequence
from tensorflow.keras.preprocessing.text import Tokenizer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

# **Prepare Train Test Sets**

In [3]:
# read Train/Test files
# split train_data into tran and validation sets with validations size = 20%
# drop PhraseId and SentenceId columns as they do not contains any imprtant data

train_data = pd.read_csv('/content/train.tsv', sep='\t')
test_data = pd.read_csv('/content/test.tsv', sep='\t')
tr_data = train_data.drop(['PhraseId','SentenceId'],axis=1)
train_set, val_set = train_test_split(tr_data,test_size=0.2,random_state=42)
train_x = train_set['Phrase']
val_x = val_set['Phrase']
train_y = train_set['Sentiment']
val_y = val_set['Sentiment']

In [5]:
train_y.value_counts()

,count
Sentiment,
2,63943
3,26220
1,21746
4,7283
0,5656


# **Load GLoVe Pretrained Embeding Model**

In [4]:
%%capture
! pip install gensim

In [5]:
import gensim.downloader as api
wv = api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


# **Config**

In [32]:
VOCAB_SIZE    = 20000
MAX_LEN       = 50
EMBEDDING_DIM = 100
HIDDEN_DIM    = 128
NUM_LAYERS    = 3
NUM_CLASSES   = 5
DROPOUT       = 0.0
BATCH_SIZE    = 64
EPOCHS        = 50
LR            = 1e-4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

# **Tokenizer (TEXT → IDS)**

In [8]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<UNK>")
tokenizer.fit_on_texts(train_x)   # fit only on train, not val/test

train_sequences = tokenizer.texts_to_sequences(train_x)
val_sequences   = tokenizer.texts_to_sequences(val_x)

train_padded = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
val_padded   = pad_sequences(val_sequences,   maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Train shape : {train_padded.shape}")
print(f"Val shape   : {val_padded.shape}")

Train shape : (124848, 50)
Val shape   : (31212, 50)


# **Build Embedding Matrix**

In [9]:
print("\nLoading GloVe vectors …")

def build_embedding_matrix(tokenizer, wv, vocab_size, embedding_dim):
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        if word in wv:
            embedding_matrix[i] = wv[word]
    return embedding_matrix

embedding_matrix = build_embedding_matrix(tokenizer, wv, VOCAB_SIZE, EMBEDDING_DIM)
print(f"Embedding matrix shape: {embedding_matrix.shape}")


Loading GloVe vectors …
Embedding matrix shape: (20000, 100)


In [41]:
embedding_matrix.shape

(100000, 100)

# **DataSet class**

In [10]:
class PhraseDataset(Dataset):
    def __init__(self, sequences, labels=None):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels.values, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.sequences[idx], self.labels[idx]
        return self.sequences[idx]

In [11]:
train_dataset = PhraseDataset(train_padded, train_y)
val_dataset   = PhraseDataset(val_padded,   val_y)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)


# **RNN Model**

In [13]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix,
                 hidden_dim, num_layers, num_classes, dropout):
        super().__init__()

        # Pretrained GloVe — frozen
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = False

        self.rnn = nn.RNN(
            input_size    = embedding_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            nonlinearity  = 'tanh',
        )

        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim*2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # Compute real lengths to skip padding in RNN
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()

        emb    = self.dropout(self.embedding(x))                                        # (batch, seq_len, embed_dim)
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)

        _, hidden = self.rnn(packed)                                                    # hidden: (num_layers*2, batch, hidden_dim)

        # Last layer: forward and backward hidden states
        forward_h  = hidden[-2]                                                         # (batch, hidden_dim)
        backward_h = hidden[-1]                                                         # (batch, hidden_dim)
        combined   = torch.cat([forward_h, backward_h], dim=1)                         # (batch, hidden_dim*2)

        return self.fc(self.dropout(combined))                                          # (batch, num_classes)


model = SentimentRNN(
    vocab_size       = VOCAB_SIZE,
    embedding_dim    = EMBEDDING_DIM,
    embedding_matrix = embedding_matrix,
    hidden_dim       = HIDDEN_DIM,
    num_layers       = NUM_LAYERS,
    num_classes      = NUM_CLASSES,
    dropout          = 0.1,
).to(DEVICE)

# **LSTM Model**

In [36]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix,
                 hidden_dim, num_layers, num_classes, dropout):
        super().__init__()

        # Pretrained GloVe — frozen
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = True

        # 🔥 RNN → LSTM
        self.lstm = nn.LSTM(
            input_size    = embedding_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # lengths for packing
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()

        emb = self.dropout(self.embedding(x))

        packed = pack_padded_sequence(
            emb,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        # 🔥 LSTM returns (output, (hidden, cell))
        _, (hidden, cell) = self.lstm(packed)

        # hidden: (num_layers*2, batch, hidden_dim)

        # last layer forward/backward
        forward_h  = hidden[-2]   # (batch, hidden_dim)
        backward_h = hidden[-1]   # (batch, hidden_dim)

        combined = torch.cat([forward_h, backward_h], dim=1)

        return self.fc(self.dropout(combined))                                       # (batch, num_classes)


model_model = SentimentLSTM(
    vocab_size       = VOCAB_SIZE,
    embedding_dim    = EMBEDDING_DIM,
    embedding_matrix = embedding_matrix,
    hidden_dim       = HIDDEN_DIM,
    num_layers       = NUM_LAYERS,
    num_classes      = NUM_CLASSES,
    dropout          = DROPOUT,
).to(DEVICE)

# **Train Loop and evaluation**

In [14]:
print(f"\nDevice          : {DEVICE}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ── 6. Training Loop ───────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(dim=1) == y).sum().item()
        total      += len(y)

    return total_loss / total, correct / total


Device          : cuda
Trainable params: 158,981


In [15]:
@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss   = criterion(logits, y)

        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(dim=1) == y).sum().item()
        total      += len(y)

    return total_loss / total, correct / total


In [37]:

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5,
)
criterion = nn.CrossEntropyLoss()

In [38]:
best_val_acc = 0.0
print(f"\n{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-" * 55)

for epoch in range(1, 50 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model, val_loader,   criterion)

    scheduler.step(vl_loss)

    flag = ""
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        # torch.save(model.state_dict(), "best_model.pt")
        flag = "  ← saved"

    print(f"{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {vl_loss:>8.4f} | {vl_acc:>7.4f}{flag}")

torch.save(model.state_dict(), "BiRNN.pt")
print(f"\nBest val accuracy: {best_val_acc:.4f}")


Epoch | Train Loss | Train Acc | Val Loss | Val Acc
-------------------------------------------------------
    1 |     0.6643 |    0.7241 |   0.8286 |  0.6695  ← saved
    2 |     0.6651 |    0.7249 |   0.8286 |  0.6675
    3 |     0.6624 |    0.7261 |   0.8308 |  0.6680
    4 |     0.6648 |    0.7249 |   0.8282 |  0.6681
    5 |     0.6638 |    0.7251 |   0.8286 |  0.6672
    6 |     0.6642 |    0.7246 |   0.8273 |  0.6683
    7 |     0.6621 |    0.7243 |   0.8289 |  0.6687
    8 |     0.6625 |    0.7266 |   0.8302 |  0.6669
    9 |     0.6603 |    0.7257 |   0.8332 |  0.6693
   10 |     0.6591 |    0.7270 |   0.8299 |  0.6695  ← saved
   11 |     0.6569 |    0.7279 |   0.8300 |  0.6681
   12 |     0.6582 |    0.7264 |   0.8300 |  0.6692
   13 |     0.6569 |    0.7285 |   0.8301 |  0.6696  ← saved
   14 |     0.6562 |    0.7297 |   0.8297 |  0.6690
   15 |     0.6551 |    0.7286 |   0.8302 |  0.6706  ← saved
   16 |     0.6559 |    0.7280 |   0.8301 |  0.6690
   17 |     0.6550 |   

# **Generate Submission**

In [52]:
test_na_rows = test_data['Phrase'].isna()
test_data.loc[test_na_rows, 'Phrase'] = ''

test_sequences  = tokenizer.texts_to_sequences(test_data['Phrase'])
test_padded  = pad_sequences(test_sequences,  maxlen=MAX_LEN, padding='post', truncating='post')

In [55]:
class TestDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = torch.tensor(sequences, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

In [56]:
test_dataset = TestDataset(test_padded)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [57]:
model.eval()
@torch.no_grad()
def predict(model, loader):
    preds = []

    for x in loader:
        x = x.to(DEVICE)

        logits = model(x)
        batch_preds = torch.argmax(logits, dim=1)

        preds.extend(batch_preds.cpu().numpy())

    return preds

In [58]:
predictions = predict(model, test_loader)

In [59]:
submission = pd.DataFrame({
    "PhraseId": test_data["PhraseId"],
    "Sentiment": predictions
})

In [61]:
submission.to_csv("submission.csv", index=False)